# 07 — Evaluate on the held-out season & register the champion

The fairest test of a game-prediction model is a **season it never saw during training**.
We load each of the three logged models, score them on `NBA_HOLDOUT_SEASON`, pick the
**champion** by hold-out log-loss, and register it to **Unity Catalog** with `@champion` and
`@challenger` aliases (the challenger is the best model of a *different* family — handy for
A/B comparisons).

We also compare against the **home-court baseline** ("always pick the home team"): the model
has to beat just predicting home every time.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os
import pandas as pd

from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score, accuracy_score

import mlflow
from mlflow import MlflowClient

load_dotenv()
spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG  = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA   = os.getenv("UC_SCHEMA",  "basketball_reference_waf")
GOLD_SCHEMA = f"{UC_SCHEMA}_gold"
FEATURES    = f"{UC_CATALOG}.{GOLD_SCHEMA}.game_features"
MODEL_NAME  = os.getenv("MODEL_NAME", "home_win_classifier")
FULL_MODEL  = f"{UC_CATALOG}.{GOLD_SCHEMA}.{MODEL_NAME}"
EXPERIMENT  = os.getenv("MLFLOW_EXPERIMENT_NAME", "/Users/alexander.booth@databricks.com/basketball-reference-waf")
HOLDOUT     = int(os.getenv("NBA_HOLDOUT_SEASON", "2025"))

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
exp = mlflow.set_experiment(EXPERIMENT)
client = MlflowClient()

FEATURE_COLS = [
    "win_pct_diff", "margin_diff", "net_rtg_diff",
    "efg_for_diff", "tov_for_diff", "orb_for_diff", "ftr_for_diff",
    "efg_against_diff", "tov_against_diff", "drb_against_diff", "ftr_against_diff",
    "rest_diff",
]
print("Registering to:", FULL_MODEL)
print("Holdout season:", HOLDOUT)

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


Registering to: alexander_booth.basketball_reference_waf_gold.home_win_classifier
Holdout season: 2025


## Load the holdout season and the three most recent runs

In [2]:
pdf = spark.table(FEATURES).toPandas()
hold = pdf[pdf["season"] == HOLDOUT].copy()
X_h = hold[FEATURE_COLS].astype(float)
y_h = hold["home_win"].astype(int).to_numpy()
print(f"Holdout games: {len(hold)} | home-win rate: {y_h.mean():.3f}")

runs = mlflow.search_runs(experiment_ids=[exp.experiment_id],
                          filter_string="tags.demo = 'home_win'",
                          order_by=["start_time DESC"])
# keep the most recent run per model family
latest = runs.drop_duplicates(subset=["tags.model_family"], keep="first")
print("\nCandidate runs:")
print(latest[["run_id", "tags.model_family", "tags.mlflow.runName"]].to_string(index=False))

Holdout games: 1075 | home-win rate: 0.542

Candidate runs:
                          run_id tags.model_family tags.mlflow.runName
25b04d684ba04f23a92ea8efa13587e7           xgboost          xgboost_v1
0cf1ba1acf1f4ad792e67d0e68e509dc     random_forest    random_forest_v1
10827692e0964dbe856d832d57039a8f            logreg           logreg_v1


## Score each model on the holdout season

In [3]:
rows = []
for _, r in latest.iterrows():
    run_id = r["run_id"]; family = r["tags.model_family"]
    model = mlflow.pyfunc.load_model(f"runs:/{run_id}/model")
    proba = model.predict(X_h)
    proba = proba.values.reshape(-1) if hasattr(proba, "values") else proba
    m = {
        "run_id": run_id, "family": family,
        "holdout_log_loss": log_loss(y_h, proba),
        "holdout_brier":    brier_score_loss(y_h, proba),
        "holdout_roc_auc":  roc_auc_score(y_h, proba),
        "holdout_accuracy": accuracy_score(y_h, (proba >= 0.5).astype(int)),
    }
    rows.append(m)
    # write the holdout metrics back onto the original run
    with mlflow.start_run(run_id=run_id):
        mlflow.log_metrics({k: v for k, v in m.items() if k.startswith("holdout_")})

board = pd.DataFrame(rows).sort_values("holdout_log_loss").reset_index(drop=True)
baseline_acc = max(y_h.mean(), 1 - y_h.mean())
print(f"Home-court baseline accuracy (always pick home): {baseline_acc:.3f}\n")
print(board.to_string(index=False))

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🏃 View run xgboost_v1 at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/112419715817170/runs/25b04d684ba04f23a92ea8efa13587e7
🧪 View experiment at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/112419715817170


🏃 View run random_forest_v1 at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/112419715817170/runs/0cf1ba1acf1f4ad792e67d0e68e509dc
🧪 View experiment at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/112419715817170


🏃 View run logreg_v1 at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/112419715817170/runs/10827692e0964dbe856d832d57039a8f
🧪 View experiment at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/112419715817170
Home-court baseline accuracy (always pick home): 0.542

                          run_id        family  holdout_log_loss  holdout_brier  holdout_roc_auc  holdout_accuracy
10827692e0964dbe856d832d57039a8f        logreg          0.606013       0.210049         0.725338          0.657674
0cf1ba1acf1f4ad792e67d0e68e509dc random_forest          0.614597       0.213029         0.720896          0.661395
25b04d684ba04f23a92ea8efa13587e7       xgboost          0.639270       0.221200         0.702680          0.655814


## Register champion + challenger to Unity Catalog

In [4]:
champion = board.iloc[0]
others = board[board["family"] != champion["family"]]
challenger = others.iloc[0] if len(others) else board.iloc[1]

def register(run_id, family):
    mv = mlflow.register_model(model_uri=f"runs:/{run_id}/model", name=FULL_MODEL,
                               tags={"model_family": family, "source_run": run_id})
    return mv

champ_mv = register(champion["run_id"], champion["family"])
chal_mv  = register(challenger["run_id"], challenger["family"])

client.set_registered_model_alias(FULL_MODEL, "champion", champ_mv.version)
client.set_registered_model_alias(FULL_MODEL, "challenger", chal_mv.version)
client.update_registered_model(
    name=FULL_MODEL,
    description=("Predicts the probability that the HOME team wins an NBA regular-season "
                 "game, from each team's season-to-date form (Four Factors, net rating, "
                 "win %, rest). Trained on Basketball Reference team game logs."),
)

print(f"champion   : {champion['family']:<14} v{champ_mv.version}  "
      f"holdout acc={champion['holdout_accuracy']:.3f}  logloss={champion['holdout_log_loss']:.4f}")
print(f"challenger : {challenger['family']:<14} v{chal_mv.version}  "
      f"holdout acc={challenger['holdout_accuracy']:.3f}")
lift = champion["holdout_accuracy"] - baseline_acc
print(f"\nChampion vs home-court baseline: {champion['holdout_accuracy']:.3f} vs {baseline_acc:.3f} "
      f"({'+' if lift >= 0 else ''}{lift*100:.1f} pts)")

Successfully registered model 'alexander_booth.basketball_reference_waf_gold.home_win_classifier'.


Uploading artifacts:   0%|          | 0/7 [00:00<?, ?it/s]

Uploading artifacts:  14%|█▍        | 1/7 [00:00<00:02,  2.54it/s]

Uploading artifacts:  14%|█▍        | 1/7 [00:00<00:02,  2.54it/s]

Uploading artifacts:  29%|██▊       | 2/7 [00:00<00:01,  2.54it/s]

...

Uploading artifacts:  86%|████████▌ | 6/7 [00:00<00:00,  5.89it/s]

Uploading artifacts: 100%|██████████| 7/7 [00:00<00:00,  5.89it/s]

Uploading artifacts: 100%|██████████| 7/7 [00:00<00:00,  7.71it/s]

Created version '2' of model 'alexander_booth.basketball_reference_waf_gold.home_win_classifier'.


champion   : logreg         v1  holdout acc=0.658  logloss=0.6060
challenger : random_forest  v2  holdout acc=0.661

Champion vs home-court baseline: 0.658 vs 0.542 (+11.5 pts)


Champion is registered in Unity Catalog with `@champion` / `@challenger` aliases.
**Next:** `08_batch_inference` scores the holdout season through the champion alias.